Scientific source notebook for the isolated reproduction tree. Calculation cells and saved execution objects are preserved. Narrative navigation has been adapted for this package. Use the reproduction runner and frozen-input prerequisites before executing. Historical evaluation blocks remain reused evidence; this copy does not establish independent replication.


# 03 · Standard factor data

**Question:** What risks does the baseline model already represent?

French FF5 and daily momentum ZIP files from vintage 202608 are parsed to decimal daily returns. RF is the daily risk-free rate, while Mkt-RF already subtracts it.

Research report · Method and code guide · Evidence map

**Collection record:** original execution outputs are retained. Read this notebook for provenance; use the offline reproduction command to rebuild the research without refreshing downloads.


**Chapter route**

Measurement and interpretation principles · Read and run this notebook · Calculation source: foundation.calendar · Results, examples and interpretation · dates and units · join the momentum factor · Findings · Analysis and takeaways · Bridge through the study


## Measurement and interpretation principles

Convert source percentage returns to decimals once. Excess stock return is $r_i-r_f$, while Mkt-RF is already an excess return. FF5 represents market, size, value, profitability and investment; the momentum factor is a daily long-short portfolio return, distinct from a stock's trailing own-price momentum.


## Read and run this notebook

The calculation definitions now live here, alongside the purpose and interpretation. Use the chapter route for the argument and the bold function names for individual definitions. Definition cells do not download data or run the full study. The results/examples follow the definitions. Save edits and restart the kernel before a full rerun so shared imports read the saved version. The original collection cells below remain archival; do not rerun them merely to read the calculations.


## Calculation source: foundation.calendar

These are the actual definitions used by the study. Read the purpose above, then follow the functions below before the results. The small Python adapter imports only these marked cells; it does not run collection or display cells.


### Imports, conventions and measurement settings

Imports and fixed conventions used by the calculations below.


In [1]:
"""Study calendar: trading sessions, quarterly formation dates, measurement cutoffs, windows and outcome quarters.

Sessions come from the price index, never from an exchange calendar library, so the calendar is the data's calendar.
A formation date is the final session of a calendar quarter. Its outcome quarter is the next calendar quarter, which
falls in one of three evaluation blocks (development, validation, confirmation) or is labelled incomplete.

Completeness of a quarter is judged on calendar dates: a quarter is complete for a data source when the source's last
date is on or after the quarter's last calendar day. Both study cutoffs (factors 2026-08-31, prices 2026-09-30) are
calendar month ends, so this is exact here.
"""

import pandas as pd

from quiet_index_500 import settings
from quiet_index_500.foundation import loader


**trading_sessions**

Every session of the price panel on or before the cutoff, as a DatetimeIndex.


In [2]:
def trading_sessions(through=settings.research_price_cutoff, apply_corrections=True):
    """Every session of the price panel on or before the cutoff, as a DatetimeIndex."""
    return loader.load_price_panels(through=through, apply_corrections=apply_corrections).adjusted_close.index


**_calendar_quarter_is_complete**

A calendar quarter is complete when the data run to its last calendar day or later.


In [3]:
def _calendar_quarter_is_complete(quarter, data_last_date):
    """A calendar quarter is complete when the data run to its last calendar day or later."""
    return quarter.end_time.normalize() <= pd.Timestamp(data_last_date)


**formation_dates**

The last session of each complete calendar quarter, from first_formation_date to last_date (a DatetimeIndex).


In [4]:
def formation_dates(sessions, first_formation_date=settings.first_formation_date, last_date=settings.research_price_cutoff):
    """The last session of each complete calendar quarter, from first_formation_date to last_date (a DatetimeIndex)."""
    sessions = pd.DatetimeIndex(sessions)
    sessions_in_range = sessions[(sessions >= pd.Timestamp(first_formation_date)) & (sessions <= pd.Timestamp(last_date))]
    last_session_by_quarter = pd.Series(sessions_in_range, index=sessions_in_range.to_period("Q")).groupby(level=0).max()
    complete_quarters = [quarter for quarter in last_session_by_quarter.index if _calendar_quarter_is_complete(quarter, last_date)]  # a partial quarter has no formation yet
    return pd.DatetimeIndex(last_session_by_quarter[complete_quarters].to_numpy(), name="formation_date")


**measurement_cutoff**

Cutoff for factor-dependent estimates: min(formation date, factor cutoff).


In [5]:
def measurement_cutoff(formation_date, factor_cutoff):
    """Cutoff for factor-dependent estimates: min(formation date, factor cutoff)."""
    return min(pd.Timestamp(formation_date), pd.Timestamp(factor_cutoff))


**estimation_window**

The window_sessions sessions ending on the last session on or before cutoff, inclusive.

Fewer sessions are returned when the history is shorter; eligibility counts then decide whether that is enough.


In [6]:
def estimation_window(sessions, cutoff, window_sessions):
    """The window_sessions sessions ending on the last session on or before cutoff, inclusive.

    Fewer sessions are returned when the history is shorter; eligibility counts then decide whether that is enough.
    """
    sessions = pd.DatetimeIndex(sessions)
    last_position = sessions.searchsorted(pd.Timestamp(cutoff), side="right") - 1  # the last session not after the cutoff
    if last_position < 0:
        return sessions[:0]
    return sessions[max(0, last_position - window_sessions + 1): last_position + 1]


**outcome_quarter**

The calendar quarter after the formation date's quarter (a pandas Period).


In [7]:
def outcome_quarter(formation_date):
    """The calendar quarter after the formation date's quarter (a pandas Period)."""
    return pd.Timestamp(formation_date).to_period("Q") + 1


**following_quarter_sessions**

Sessions of the outcome quarter that are present in sessions (empty if none are loaded yet).


In [8]:
def following_quarter_sessions(sessions, formation_date):
    """Sessions of the outcome quarter that are present in sessions (empty if none are loaded yet)."""
    sessions = pd.DatetimeIndex(sessions)
    quarter = outcome_quarter(formation_date)
    return sessions[(sessions >= quarter.start_time) & (sessions <= quarter.end_time)]


**outcome_block**

development, validation or confirmation by the outcome quarter (docs/STUDY_SPEC.md section 2), else incomplete.

Outcome quarters after the confirmation block (2026 Q3 and later) are incomplete for the evaluation. An outcome
quarter before the development block means a formation date earlier than the study allows, which is an error.


In [9]:
def outcome_block(formation_date):
    """development, validation or confirmation by the outcome quarter (docs/STUDY_SPEC.md section 2), else incomplete.

    Outcome quarters after the confirmation block (2026 Q3 and later) are incomplete for the evaluation. An outcome
    quarter before the development block means a formation date earlier than the study allows, which is an error.
    """
    quarter = outcome_quarter(formation_date)
    for block_name, (first_quarter, last_quarter) in [("development", settings.development_outcome_quarters),
                                                      ("validation", settings.validation_outcome_quarters),
                                                      ("confirmation", settings.confirmation_outcome_quarters)]:
        if pd.Period(first_quarter, "Q") <= quarter <= pd.Period(last_quarter, "Q"):
            return block_name
    if quarter < pd.Period(settings.development_outcome_quarters[0], "Q"):
        raise ValueError(f"formation {pd.Timestamp(formation_date).date()} is before the first formation {settings.first_formation_date}")
    return "incomplete"


**following_quarter_complete**

Whether the whole outcome quarter is covered by data ending on data_last_date.


In [10]:
def following_quarter_complete(formation_date, data_last_date):
    """Whether the whole outcome quarter is covered by data ending on data_last_date."""
    return _calendar_quarter_is_complete(outcome_quarter(formation_date), data_last_date)


**formation_calendar**

One row per formation date: outcome quarter and block, both measurement cutoffs, outcome completeness and session range.


In [11]:
def formation_calendar(sessions, factor_cutoff, price_cutoff=settings.research_price_cutoff):
    """One row per formation date: outcome quarter and block, both measurement cutoffs, outcome completeness and session range."""
    sessions = pd.DatetimeIndex(sessions)
    rows = []
    for formation_date in formation_dates(sessions, last_date=price_cutoff):
        outcome_sessions = following_quarter_sessions(sessions[sessions <= pd.Timestamp(price_cutoff)], formation_date)
        rows.append({
            "formation_date": formation_date,
            "outcome_quarter": str(outcome_quarter(formation_date)),
            "outcome_block": outcome_block(formation_date),
            "factor_measurement_cutoff": measurement_cutoff(formation_date, factor_cutoff),
            "price_measurement_cutoff": formation_date,  # size, liquidity and price-only measures use the formation date itself
            "outcome_complete_for_factor_results": following_quarter_complete(formation_date, factor_cutoff),
            "outcome_complete_for_price_results": following_quarter_complete(formation_date, price_cutoff),
            "outcome_first_session": outcome_sessions.min() if len(outcome_sessions) else pd.NaT,
            "outcome_last_session": outcome_sessions.max() if len(outcome_sessions) else pd.NaT,
            "outcome_sessions": len(outcome_sessions),
        })
    return pd.DataFrame(rows).set_index("formation_date")


## Results, examples and interpretation

The cells below use the definitions above and the frozen study inputs.


In [1]:
import hashlib
import io
import json
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

ff5_vintage = "202608"  # CRSP vintage the French library built the file from; a new download that differs gets a new name
price_vintage_last_session = "2026-10-12"  # must match notebooks/02_prices.ipynb; the study's sessions come from that pull

repo_root_folder = Path("..").resolve()
raw_data_folder = Path("../data/raw")
processed_data_folder = Path("../data/processed")
ff5_source_url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip"
ff5_zip_path = raw_data_folder / f"ff5_factors_daily_crsp_{ff5_vintage}.zip"
ff5_manifest_path = raw_data_folder / f"ff5_factors_daily_crsp_{ff5_vintage}.manifest.json"

if not ff5_manifest_path.exists():  # download once; the library revises history when it rebuilds, so each vintage is kept as its own file
    response = requests.get(ff5_source_url, timeout=60)
    response.raise_for_status()
    zip_bytes = response.content
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
        member = archive.infolist()[0]
        first_line = archive.read(member).decode("utf-8").splitlines()[0]
    assert f"{ff5_vintage} CRSP" in first_line, f"downloaded file says: {first_line}; set ff5_vintage to match and rerun"
    ff5_zip_path.write_bytes(zip_bytes)
    ff5_manifest = {
        "source": "Kenneth R. French Data Library, Fama/French 5 Factors (2x3), daily",
        "source_url": ff5_source_url,
        "vintage_line": first_line.strip(),
        "csv_member": member.filename,
        "csv_timestamp_inside_zip": datetime(*member.date_time).isoformat(),
        "http_last_modified": response.headers.get("Last-Modified"),
        "pulled_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        "file": ff5_zip_path.resolve().relative_to(repo_root_folder).as_posix(),
        "sha256": hashlib.sha256(zip_bytes).hexdigest(),
        "bytes": len(zip_bytes),
        "note": "Values in the source are percent per day. Public data; the zip is committed as downloaded.",
    }
    ff5_manifest_path.write_text(json.dumps(ff5_manifest, indent=2) + "\n")

json.loads(ff5_manifest_path.read_text())

{'source': 'Kenneth R. French Data Library, Fama/French 5 Factors (2x3), daily',
 'source_url': 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip',
 'vintage_line': 'This file was created by using the 202608 CRSP database.',
 'csv_member': 'F-F_Research_Data_5_Factors_2x3_daily.csv',
 'csv_timestamp_inside_zip': '2026-09-24T09:09:46',
 'http_last_modified': 'Fri, 25 Sep 2026 21:55:02 GMT',
 'pulled_at_utc': '2026-10-14T01:54:01+00:00',
 'file': 'data/raw/ff5_factors_daily_crsp_202608.zip',
 'sha256': '7881223b51929ee0d6d23b8a28524aaec478a2c3d4eeca47224955cc845c5186',
 'bytes': 150292,
 'note': 'Values in the source are percent per day. Public data; the zip is committed as downloaded.'}

## dates and units

The source percentage return is divided by 100 exactly once, then aligned to the price calendar. Do not subtract RF from Mkt-RF again.


In [2]:
with zipfile.ZipFile(ff5_zip_path) as archive:
    ff5_csv_lines = archive.read(archive.infolist()[0]).decode("utf-8").splitlines()

header_line_number = next(i for i, line in enumerate(ff5_csv_lines) if line.startswith(",Mkt-RF"))  # preamble lines sit above the column header
footer_line_number = next(i for i in range(header_line_number + 1, len(ff5_csv_lines)) if not ff5_csv_lines[i].strip())  # a blank line, then the copyright line

ff5_daily_percent = pd.read_csv(io.StringIO("\n".join(ff5_csv_lines[header_line_number:footer_line_number])), index_col=0)
ff5_daily_percent.index = pd.to_datetime(ff5_daily_percent.index.astype(str), format="%Y%m%d")

ff5_daily_all_history = (ff5_daily_percent / 100).rename(columns={  # percent per day becomes decimal, once; nothing downstream rescales
    "Mkt-RF": "mkt_rf", "SMB": "smb", "HML": "hml", "RMW": "rmw", "CMA": "cma", "RF": "rf"}).rename_axis("date")

study_sessions = pd.read_csv(raw_data_folder / f"prices_adjusted_close_through_{price_vintage_last_session}.csv", usecols=["date"], parse_dates=["date"])["date"]
factor_data_last_date = ff5_daily_all_history.index.max()            # set by the French library's publishing lag, read from the data
estimation_cutoff = min(factor_data_last_date, study_sessions.max())  # the last date on which both prices and factors exist

ff5_daily = ff5_daily_all_history.loc[study_sessions.min():estimation_cutoff]
study_sessions_to_cutoff = pd.DatetimeIndex(study_sessions[study_sessions <= estimation_cutoff])

assert ff5_daily.notna().all().all()
assert ff5_daily.index.is_unique and ff5_daily.index.is_monotonic_increasing
assert ff5_daily.index.equals(study_sessions_to_cutoff)  # every French trading day is a price session and every price session to the cutoff has factors
assert ff5_daily["mkt_rf"].abs().max() < 0.25 and ff5_daily["rf"].between(0, 0.001).all()  # decimal units: no daily market move beyond 25%, rf a small daily rate

ff5_daily_path = processed_data_folder / f"ff5_factors_daily_{ff5_daily.index.min():%Y-%m-%d}_{ff5_daily.index.max():%Y-%m-%d}.csv"
ff5_daily.to_csv(ff5_daily_path)

ff5_daily_read_back = pd.read_csv(ff5_daily_path)
ff5_daily_manifest = {
    "file": ff5_daily_path.resolve().relative_to(repo_root_folder).as_posix(),
    "built_by": "notebooks/03_factors.ipynb",
    "built_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "inputs": [{"file": p.resolve().relative_to(repo_root_folder).as_posix(), "sha256": hashlib.sha256(p.read_bytes()).hexdigest()}
               for p in [ff5_zip_path, raw_data_folder / f"prices_adjusted_close_through_{price_vintage_last_session}.csv"]],
    "sha256": hashlib.sha256(ff5_daily_path.read_bytes()).hexdigest(),
    "rows": len(ff5_daily_read_back),
    "columns": ff5_daily_read_back.columns.tolist(),
    "factor_data_last_date": f"{factor_data_last_date:%Y-%m-%d}",
    "estimation_cutoff": f"{estimation_cutoff:%Y-%m-%d}",
    "note": "Fama-French five factors (2x3), daily, decimal units. mkt_rf is the CRSP value-weighted market minus rf, not SPY. "
            "smb is the five-factor definition. rf is the one-month T-bill as a simple daily rate. Rows run from the first study session "
            "to the estimation cutoff, the earlier of the last factor date and the last price session.",
}
ff5_daily_path.with_suffix(".manifest.json").write_text(json.dumps(ff5_daily_manifest, indent=2) + "\n")

ff5_daily.describe()

,mkt_rf,smb,hml,rmw,cma,rf
count,2932.000000,2932.000000,2932.000000,2932.000000,2932.000000,2932.000000
mean,0.000496,-0.000071,-0.000030,0.000077,-0.000040,0.000083
std,0.011418,0.006772,0.008800,0.005583,0.004907,0.000084
min,-0.120100,-0.045800,-0.050300,-0.022300,-0.029100,0.000000
25%,-0.004000,-0.004025,-0.004600,-0.003000,-0.002700,0.000000
50%,0.000600,-0.000300,-0.000350,-0.000100,-0.000200,0.000100
75%,0.006025,0.003700,0.004325,0.003100,0.002425,0.000200
max,0.096500,0.057100,0.067300,0.042500,0.024700,0.000200


In [3]:
momentum_source_url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Momentum_Factor_daily_CSV.zip"
momentum_zip_path = raw_data_folder / f"momentum_factor_daily_crsp_{ff5_vintage}.zip"
momentum_manifest_path = raw_data_folder / f"momentum_factor_daily_crsp_{ff5_vintage}.manifest.json"

if not momentum_manifest_path.exists():  # download once; same vintage as the five factors, so both tables come from one CRSP build
    response = requests.get(momentum_source_url, timeout=60)
    response.raise_for_status()
    zip_bytes = response.content
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
        member = archive.infolist()[0]
        first_line = archive.read(member).decode("utf-8").splitlines()[0]
    assert f"{ff5_vintage} CRSP" in first_line, f"downloaded file says: {first_line}; the momentum and five-factor vintages must match"
    momentum_zip_path.write_bytes(zip_bytes)
    momentum_manifest = {
        "source": "Kenneth R. French Data Library, Momentum Factor (Mom), daily",
        "source_url": momentum_source_url,
        "vintage_line": first_line.strip(),
        "csv_member": member.filename,
        "csv_timestamp_inside_zip": datetime(*member.date_time).isoformat(),
        "http_last_modified": response.headers.get("Last-Modified"),
        "pulled_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        "file": momentum_zip_path.resolve().relative_to(repo_root_folder).as_posix(),
        "sha256": hashlib.sha256(zip_bytes).hexdigest(),
        "bytes": len(zip_bytes),
        "note": "Values in the source are percent per day; missing values are -99.99 or -999. Prior return runs from day -250 to day -21. "
                "Public data; the zip is committed as downloaded.",
    }
    momentum_manifest_path.write_text(json.dumps(momentum_manifest, indent=2) + "\n")

json.loads(momentum_manifest_path.read_text())

{'source': 'Kenneth R. French Data Library, Momentum Factor (Mom), daily',
 'source_url': 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Momentum_Factor_daily_CSV.zip',
 'vintage_line': 'This file was created by using the 202608 CRSP database.  It',
 'csv_member': 'F-F_Momentum_Factor_daily.csv',
 'csv_timestamp_inside_zip': '2026-09-24T09:14:28',
 'http_last_modified': 'Fri, 25 Sep 2026 21:55:02 GMT',
 'pulled_at_utc': '2026-10-14T02:10:28+00:00',
 'file': 'data/raw/momentum_factor_daily_crsp_202608.zip',
 'sha256': 'ecf82b116e6ec56d2b79a2f27bd1d1598c217b5c6c25caa55b945bdd8a2c0b78',
 'bytes': 85269,
 'note': 'Values in the source are percent per day; missing values are -99.99 or -999. Prior return runs from day -250 to day -21. Public data; the zip is committed as downloaded.'}

## join the momentum factor

The left join preserves FF5 dates and asserts momentum coverage. The factor is a daily long-short portfolio return, not an individual stock price ratio.


In [4]:
with zipfile.ZipFile(momentum_zip_path) as archive:
    momentum_csv_lines = archive.read(archive.infolist()[0]).decode("utf-8").splitlines()

momentum_header_line_number = next(i for i, line in enumerate(momentum_csv_lines) if line.replace(" ", "").startswith(",Mom"))
momentum_footer_line_number = next(i for i in range(momentum_header_line_number + 1, len(momentum_csv_lines)) if not momentum_csv_lines[i].strip())

momentum_daily_percent = pd.read_csv(io.StringIO("\n".join(momentum_csv_lines[momentum_header_line_number:momentum_footer_line_number])),
                                     index_col=0, na_values=["-99.99", "-999"])
momentum_daily_percent.index = pd.to_datetime(momentum_daily_percent.index.astype(str), format="%Y%m%d")
momentum_daily_percent.columns = momentum_daily_percent.columns.str.strip()

momentum_daily_all_history = (momentum_daily_percent["Mom"] / 100).rename("mom").rename_axis("date")  # percent per day becomes decimal, once

ff5_mom_daily = ff5_daily.join(momentum_daily_all_history, how="left")  # the five-factor table's dates decide the rows

assert ff5_mom_daily["mom"].notna().all()                                       # momentum exists on every five-factor date in the study window
assert momentum_daily_all_history.index.max() == factor_data_last_date          # both files end on the same day
assert ff5_mom_daily["mom"].abs().max() < 0.25                                  # decimal units

ff5_mom_daily_path = processed_data_folder / f"ff5_mom_factors_daily_{ff5_mom_daily.index.min():%Y-%m-%d}_{ff5_mom_daily.index.max():%Y-%m-%d}.csv"
ff5_mom_daily.to_csv(ff5_mom_daily_path)

ff5_mom_daily_read_back = pd.read_csv(ff5_mom_daily_path)
ff5_mom_daily_manifest = {
    "file": ff5_mom_daily_path.resolve().relative_to(repo_root_folder).as_posix(),
    "built_by": "notebooks/03_factors.ipynb",
    "built_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "inputs": [{"file": p.resolve().relative_to(repo_root_folder).as_posix(), "sha256": hashlib.sha256(p.read_bytes()).hexdigest()}
               for p in [ff5_zip_path, momentum_zip_path, ff5_daily_path]],
    "sha256": hashlib.sha256(ff5_mom_daily_path.read_bytes()).hexdigest(),
    "rows": len(ff5_mom_daily_read_back),
    "columns": ff5_mom_daily_read_back.columns.tolist(),
    "factor_data_last_date": f"{factor_data_last_date:%Y-%m-%d}",
    "estimation_cutoff": f"{estimation_cutoff:%Y-%m-%d}",
    "note": "The five-factor table plus French's daily momentum factor (mom, decimal), same CRSP vintage, same rows. A separate table: "
            "the five-factor table is unchanged. French's mom sorts on prior return from day -250 to day -21; it is kept as a benchmark "
            "for the project's own 12-1 momentum.",
}
ff5_mom_daily_path.with_suffix(".manifest.json").write_text(json.dumps(ff5_mom_daily_manifest, indent=2) + "\n")

ff5_mom_daily.corr().round(2)

,mkt_rf,smb,hml,rmw,cma,rf,mom
mkt_rf,1.00,0.19,-0.10,-0.23,-0.27,-0.00,-0.10
smb,0.19,1.00,0.27,-0.25,0.05,-0.03,-0.30
hml,-0.10,0.27,1.00,0.32,0.56,-0.02,-0.30
rmw,-0.23,-0.25,0.32,1.00,0.29,-0.02,-0.10
cma,-0.27,0.05,0.56,0.29,1.00,-0.03,-0.03
rf,-0.00,-0.03,-0.02,-0.02,-0.03,1.00,0.01
mom,-0.10,-0.30,-0.30,-0.10,-0.03,0.01,1.00


## Findings

The saved factor panel has 2,932 dates through August 31, 2026. September prices therefore do not imply September factor outcomes. Own-stock momentum in Notebook 05 is a characteristic, not this traded factor.

## Analysis and takeaways

Prices extend beyond the final factor date, August 31, 2026. That gap limits aligned model observations and complete future quarters; it cannot be repaired by filling factor returns. Common dates and consistent return units come before coefficient interpretation.

## Bridge through the study

03b fixes valid measurement dates. 04 uses FF5 plus momentum as its main baseline; 05 uses RF with SPY; 08 and 09 need factors to evaluate complete future quarters.
